In [1]:
from IPython.display import HTML, display

html = r"""
<style>
  *{box-sizing:border-box;font-family:'Segoe UI',system-ui,sans-serif}
  .app{max-width:520px;margin:auto;background:#0f172a;color:#e2e8f0;border-radius:22px;padding:22px;box-shadow:0 10px 40px #0006}
  h2{margin:0 0 4px;text-align:center}
  .sub{text-align:center;color:#94a3b8;font-size:13px;margin-bottom:14px}
  .modes{display:flex;gap:8px;margin-bottom:16px}
  .mode{flex:1;padding:10px 4px;border-radius:12px;border:2px solid #334155;background:#1e293b;color:#cbd5e1;cursor:pointer;font-size:14px;transition:.2s}
  .mode.on{border-color:#22d3ee;background:#0e7490;color:#fff}
  .ring{position:relative;width:200px;height:200px;margin:8px auto}
  .ring svg{transform:rotate(-90deg)}
  .center{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center}
  .num{font-size:54px;font-weight:700}
  .goal{color:#94a3b8;font-size:13px}
  .btns{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin:14px 0}
  .btn{padding:14px;border:none;border-radius:14px;font-size:16px;font-weight:600;color:#fff;cursor:pointer;transition:.15s}
  .btn:active{transform:scale(.95)}
  .btn:disabled{opacity:.4;cursor:not-allowed}
  .add{background:#06b6d4}.sub1{background:#f59e0b}.undo{background:#6366f1}.reset{background:#ef4444}
  .goalrow{display:flex;align-items:center;justify-content:center;gap:12px;margin-bottom:10px;font-size:14px}
  .goalrow button{width:34px;height:34px;border-radius:50%;border:none;background:#334155;color:#fff;font-size:18px;cursor:pointer}
  #toast{text-align:center;min-height:24px;color:#22d3ee;font-weight:600;margin-bottom:6px}
  #log{max-height:110px;overflow:auto;font-size:13px;color:#94a3b8;background:#1e293b;border-radius:10px;padding:8px 12px}
  video{width:100%;border-radius:12px;margin-top:10px;display:none;transform:scaleX(-1)}
  .help{font-size:12px;color:#94a3b8;margin-top:10px;line-height:1.6}
</style>

<div class="app">
  <h2>💧 Water Tracker</h2>
  <div class="sub">How many glasses of water have you had today?</div>

  <div class="modes">
    <button class="mode on" id="mMouse">🖱 Mouse</button>
    <button class="mode" id="mVoice">🎤 Voice</button>
    <button class="mode" id="mGest">✋ Gesture</button>
  </div>

  <div class="ring">
    <svg width="200" height="200">
      <circle cx="100" cy="100" r="88" stroke="#1e293b" stroke-width="16" fill="none"/>
      <circle id="arc" cx="100" cy="100" r="88" stroke="#22d3ee" stroke-width="16" fill="none"
              stroke-linecap="round" stroke-dasharray="553" stroke-dashoffset="553" style="transition:.5s"/>
    </svg>
    <div class="center">
      <div class="num" id="count">0</div>
      <div class="goal" id="goalTxt">Goal: 8 glasses</div>
    </div>
  </div>

  <div id="toast"></div>

  <div class="btns">
    <button class="btn add" id="bAdd">➕ Add Glass</button>
    <button class="btn sub1" id="bSub">➖ Subtract</button>
    <button class="btn undo" id="bUndo">↩ Undo</button>
    <button class="btn reset" id="bReset">🗑 Reset</button>
  </div>

  <div class="goalrow">
    Change Goal:
    <button id="gMinus">−</button>
    <b id="goalNum">8</b>
    <button id="gPlus">+</button>
  </div>

  <div id="log">No entries yet...</div>
  <video id="cam" autoplay playsinline muted></video>

  <div class="help">
    <b>🎤 Voice Commands:</b> "plus / add" • "minus / subtract" • "undo" • "reset"<br>
    <b>✋ Gestures:</b> 1 finger = +1 • 2 fingers = −1 • 3 fingers = Undo • 4 fingers held = Reset
  </div>
</div>

<script>
(function(){
  const $ = id => document.getElementById(id);
  let count = 0, goal = 8, history = [], logs = [];

  function toast(t){
    $('toast').textContent = t;
    clearTimeout(toast.t);
    toast.t = setTimeout(() => $('toast').textContent = '', 2200);
  }

  function render(){
    $('count').textContent = count;
    $('goalTxt').textContent = 'Goal: ' + goal + ' glasses';
    $('goalNum').textContent = goal;

    const p = Math.min(count / goal, 1);
    $('arc').style.strokeDashoffset = 553 * (1 - p);
    $('arc').style.stroke = p >= 1 ? '#22c55e' : '#22d3ee';

    $('log').innerHTML = logs.length
      ? logs.slice().reverse().join('<br>')
      : 'No entries yet...';
  }

  function stamp(msg){
    logs.push(new Date().toLocaleTimeString() + ' — ' + msg);
  }

  const actions = {
    add(){
      history.push(count);
      count++;
      stamp('+1 glass');
      toast(count >= goal ? '🎉 Goal completed! Great job!' : '✅ +1 glass added');
      render();
    },

    sub(){
      if(count <= 0) return toast('The count is already 0');
      history.push(count);
      count--;
      stamp('−1 glass');
      toast('➖ 1 glass removed');
      render();
    },

    undo(){
      if(!history.length) return toast('Nothing to undo');
      count = history.pop();
      stamp('Undo');
      toast('↩ Undo successful');
      render();
    },

    reset(){
      history.push(count);
      count = 0;
      stamp('Reset');
      toast('🗑 Tracker reset');
      render();
    }
  };

  // Mouse Controls
  const actBtns = ['bAdd', 'bSub', 'bUndo', 'bReset'];

  $('bAdd').onclick = actions.add;
  $('bSub').onclick = actions.sub;
  $('bUndo').onclick = actions.undo;
  $('bReset').onclick = actions.reset;

  $('gPlus').onclick = () => {
    goal = Math.min(goal + 1, 30);
    render();
  };

  $('gMinus').onclick = () => {
    goal = Math.max(goal - 1, 1);
    render();
  };

  $('mMouse').onclick = function(){
    const on = !this.classList.contains('on');
    this.classList.toggle('on', on);
    actBtns.forEach(i => $(i).disabled = !on);
    toast(on ? '🖱 Mouse controls enabled' : '🖱 Mouse controls disabled');
  };

  // Voice Controls
  let rec = null, voiceOn = false;

  function handleVoice(t){
    t = t.toLowerCase();

    if(/reset|clear|remove all/.test(t)) return actions.reset();
    if(/undo|back/.test(t)) return actions.undo();
    if(/minus|subtract|decrease|remove/.test(t)) return actions.sub();
    if(/plus|add|increase|glass|one/.test(t)) return actions.add();

    toast('Command not recognized: "' + t + '"');
  }

  $('mVoice').onclick = function(){
    const SR = window.SpeechRecognition || window.webkitSpeechRecognition;

    if(!SR){
      return toast('Voice recognition is not supported. Please use Chrome.');
    }

    voiceOn = !voiceOn;
    this.classList.toggle('on', voiceOn);

    if(voiceOn){
      rec = new SR();
      rec.lang = 'en-US';
      rec.continuous = true;
      rec.interimResults = false;

      rec.onresult = e => {
        const t = e.results[e.results.length - 1][0].transcript.trim();
        toast('🎤 "' + t + '"');
        handleVoice(t);
      };

      rec.onerror = e => {
        if(e.error === 'not-allowed'){
          toast('Please allow microphone access');
          voiceOn = false;
          $('mVoice').classList.remove('on');
        }
      };

      rec.onend = () => {
        if(voiceOn) try { rec.start(); } catch(e) {}
      };

      try {
        rec.start();
        toast('🎤 Listening... Speak now');
      } catch(e) {}

    } else {
      rec && rec.stop();
      toast('🎤 Voice controls disabled');
    }
  };

  // Gesture Controls using MediaPipe Hands
  let gestOn = false, hands = null, stream = null;
  let lastFire = 0, lastG = -1, stable = 0, loaded = false;

  const V = $('cam');

  function loadScript(src){
    return new Promise((res, rej) => {
      const s = document.createElement('script');
      s.src = src;
      s.crossOrigin = 'anonymous';
      s.onload = res;
      s.onerror = rej;
      document.head.appendChild(s);
    });
  }

  function countFingers(lm){
    let n = 0;
    [[8,6],[12,10],[16,14],[20,18]].forEach(([t,p]) => {
      if(lm[t].y < lm[p].y) n++;
    });
    return n;
  }

  function onHands(r){
    if(!gestOn) return;

    if(!r.multiHandLandmarks || !r.multiHandLandmarks.length){
      lastG = -1;
      stable = 0;
      return;
    }

    const g = countFingers(r.multiHandLandmarks[0]);

    if(g === lastG) stable++;
    else {
      lastG = g;
      stable = 1;
    }

    const need = g === 4 ? 18 : 8;
    const now = Date.now();

    if(stable >= need && now - lastFire > 1500 && g >= 1 && g <= 4){
      lastFire = now;
      stable = 0;

      toast('✋ ' + g + ' finger(s) detected');

      ({1:actions.add, 2:actions.sub, 3:actions.undo, 4:actions.reset})[g]();
    }
  }

  async function loop(){
    if(!gestOn) return;

    if(V.readyState >= 2) await hands.send({image:V});

    requestAnimationFrame(loop);
  }

  $('mGest').onclick = async function(){
    gestOn = !gestOn;
    this.classList.toggle('on', gestOn);

    if(gestOn){
      try {
        toast('⏳ Loading camera and hand-tracking model...');

        stream = await navigator.mediaDevices.getUserMedia({
          video:{width:480, height:360}
        });

        V.srcObject = stream;
        V.style.display = 'block';

        if(!loaded){
          await loadScript(
            'https://cdn.jsdelivr.net/npm/@mediapipe/hands@0.4.1675469240/hands.js'
          );

          hands = new Hands({
            locateFile: f =>
              'https://cdn.jsdelivr.net/npm/@mediapipe/hands@0.4.1675469240/' + f
          });

          hands.setOptions({
            maxNumHands:1,
            modelComplexity:0,
            minDetectionConfidence:0.7,
            minTrackingConfidence:0.6
          });

          hands.onResults(onHands);
          loaded = true;
        }

        toast('✋ Gestures enabled — show your hand to the camera');
        loop();

      } catch(e) {
        gestOn = false;
        this.classList.remove('on');
        stream && stream.getTracks().forEach(t => t.stop());
        V.style.display = 'none';
        toast('Camera error: ' + e.message);
      }

    } else {
      stream && stream.getTracks().forEach(t => t.stop());
      V.srcObject = null;
      V.style.display = 'none';
      toast('✋ Gesture controls disabled');
    }
  };

  render();
})();
</script>
"""

display(HTML(html))